# YOLOv12 Area Attention

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/15-area-attention/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Compare full attention and contiguous-token area attention with shared Q,K,V."""
import math
import torch
from torch import nn
from torch.nn import functional as F


def attend(tokens, projection, areas):
    b, n, c = tokens.shape
    assert areas > 0 and n % areas == 0
    q, k, v = projection(tokens).chunk(3, dim=-1)
    q, k, v = [t.reshape(b * areas, n // areas, c) for t in (q, k, v)]
    weights = (q @ k.transpose(-2, -1) / math.sqrt(c)).softmax(-1)
    return (weights @ v).reshape(b, n, c), weights


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    areas = 4  # Change only this value to 2 or 1 for the area-count exercise.
    changed_index = 15  # Change only this value to 3 for the near-token exercise.
    query_index = 0
    # H=4,W=4: area=4 gives four horizontal strips, not four square windows.
    tokens = torch.arange(16.).reshape(1, 16, 1).repeat(1, 1, 2) / 16
    projection = nn.Linear(2, 6, bias=False)
    with torch.no_grad():
        projection.weight.copy_(torch.eye(2).repeat(3, 1))
    full, full_weights = attend(tokens, projection, 1)
    area, area_weights = attend(tokens, projection, areas)
    changed = tokens.clone()
    changed[:, changed_index] += 5
    full_changed, _ = attend(changed, projection, 1)
    area_changed, _ = attend(changed, projection, areas)
    area_size = tokens.shape[1] // areas
    same_area = query_index // area_size == changed_index // area_size
    full_affected = not torch.allclose(full[:, query_index], full_changed[:, query_index])
    area_affected = not torch.allclose(area[:, query_index], area_changed[:, query_index])
    assert full_affected and area_affected == same_area
    assert full_weights.numel() == tokens.shape[0] * tokens.shape[1] ** 2
    assert area_weights.numel() == tokens.shape[0] * tokens.shape[1] ** 2 // areas
    print('full affinity shape/count:', tuple(full_weights.shape), full_weights.numel())
    print('area affinity shape/count:', tuple(area_weights.shape), area_weights.numel())
    # token0 outputs here are multiples of 1/32, so 5 decimals print them exactly (no rounding ties).
    print(f'first-token output full={full[0,0,0].item():.5f}, area={area[0,0,0].item():.5f}')
    print(f'changing token {changed_index} affects token {query_index}: full={full_affected}, area={area_affected}; same area={same_area}')
    print(f'after intervention: full={full_changed[0,0,0].item():.5f}, area={area_changed[0,0,0].item():.5f}')
    optimizer = torch.optim.SGD(projection.parameters(), lr=.05)
    optimizer.zero_grad()
    loss = F.mse_loss(area, tokens)
    loss.backward()
    assert projection.weight.grad.abs().sum() > 0
    optimizer.step()
    print(f'area-attention backward/step verified; loss={loss.item():.4f}')


if __name__ == '__main__':
    main()


full affinity shape/count: (1, 16, 16) 256
area affinity shape/count: (4, 4, 4) 64
first-token output full=0.46875, area=0.09375
changing token 15 affects token 0: full=True, area=False; same area=False
after intervention: full=0.78125, area=0.09375
area-attention backward/step verified; loss=0.0048
